In [ ]:
import folium
import pandas as pd
import geopandas as gpd
import requests

In [ ]:
# Load Irish county boundaries from Tailte Eireann via data.gov.ie
# Source: https://data.gov.ie/dataset/counties-national-statutory-boundaries-20191

counties = gpd.read_file("data/counties.geojson")

In [ ]:
# Check the data loaded correctly
print(counties.head()) # Print the first few rows of counties.geojson variable
print(counties.columns) # Print column titles from counties.geojson variable

In [ ]:
# Load CSO afforestation data by county
# Source: Central Statistics Office Ireland AFA01 Afforestation Area 2023
# https://www.cso.ie

forestry = pd.read_csv("data/forestry.csv") #Create DataFrame 'forestry'

# Check the data loaded correctly
print(forestry.head())
print(forestry.columns)

In [ ]:
# Check unique county names in forestry data
print(forestry['County'].unique())
print(len(forestry['County'].unique()))

In [ ]:
# Check unique species values
print(forestry['Species'].unique())

In [ ]:
# Check unique forest owner values
print(forestry['Forest Owner'].unique())

In [ ]:
def prepare_forestry_data(df, species, owner='Total Afforestation'):
    """
    Filters and cleans the CSO forestry data for a given tree species.
    
    Removes the national total row, filters by species and forest owner,
    converts county names to uppercase to match GeoJSON format, and 
    replaces missing values with 0.
    
    Parameters:
    -----------
    df : pandas DataFrame
        Raw forestry data loaded from forestry.csv
    species : str
        Tree species to filter by e.g. 'Total Afforestation', 
        'Broadleaf', 'Conifer'
    owner : str
        Forest owner category to filter by.
        Default is 'Total Afforestation'
    
    Returns:
    --------
    pandas DataFrame
        Cleaned dataframe with one row per county
    """
    # Remove Ireland row
    df = df[df['County'] != 'Ireland']
    
    # Filter counties by species and owner
    df = df[(df['Species'] == species) & 
            (df['Forest Owner'] == owner)]
    
    # Clean county names to match GeoJSON format
    df = df.copy()
    df['County'] = df['County'].str.replace('Co. ', '').str.upper()
    
    # Replace NaN values with 0
    df['VALUE'] = df['VALUE'].fillna(0)
    
    return df

In [ ]:
# Prep data for all three map layers
total = prepare_forestry_data(forestry, 'Total Afforestation')
broadleaf = prepare_forestry_data(forestry, 'Broadleaf')
conifer = prepare_forestry_data(forestry, 'Conifer')

print("Total rows:", len(total))
print("Broadleaf rows:", len(broadleaf))
print("Conifer rows:", len(conifer))

In [ ]:
# Check county names have been standardized
print(total['County'].unique())

In [ ]:
# Check CRS of counties data
print(counties.crs)

In [ ]:
# Convert from Irish Transverse Mercator to WGS 84
counties = counties.to_crs(epsg=4326)

# Verify the conversion
print(counties.crs)

In [ ]:
# Visualize county borders
m_test = counties.explore('COUNTY', cmap='viridis')
m_test #shows the map

In [ ]:
def merge_forestry_data(counties_gdf, forestry_df, species):
    """
    Merges county boundary GeoDataFrame with cleaned forestry data.
    
    Joins county boundaries with forestry data on county name and 
    removes unneccessary columns, leaving only boundary geometry 
    and afforestation values.
    
    Parameters:
    -----------
    counties_gdf : GeoDataFrame
        County boundary data loaded from counties.geojson
    forestry_df : pandas DataFrame
        Cleaned forestry data from prepare_forestry_data function
    species : str
        Species name for the forestry data e.g. 'Total Afforestation',
        'Broadleaf', 'Conifer'
    
    Returns:
    --------
    GeoDataFrame
        Merged GeoDataFrame containing county boundaries and 
        afforestation values ready for mapping
    """
    # Merge boundaries with forestry data by county name
    merged = counties_gdf.merge(forestry_df, left_on='COUNTY', right_on='County')
    
    # Drop unneccessary columns
    merged = merged.drop(columns=['Statistic Label', 'Year', 
                                  'Species', 'Forest Owner', 
                                  'UNIT', 'County'])
    return merged

In [ ]:
# Merge county boundaries with afforestation data to create GeoDataFrames for mapping
merged_total = merge_forestry_data(counties, total, 'Total Afforestation')
merged_broadleaf = merge_forestry_data(counties, broadleaf, 'Broadleaf')
merged_conifer = merge_forestry_data(counties, conifer, 'Conifer')

# Verify merges have produced expected number of rows (31), taking into consideration the extra polgons from Dublin, Cork and Galway's city councils
print("Total rows:", len(merged_total))
print("Broadleaf rows:", len(merged_broadleaf))
print("Conifer rows:", len(merged_conifer))

In [ ]:
def add_folium_layer(merged_gdf, map_object, caption, name, cmap='YlGn'):
    """
    Adds a folium layer to an existing map.
    
    Uses geopandas explore() to visualise afforestation values 
    by county and adds the layer to an existing map object.
    
    Parameters:
    -----------
    merged_gdf : GeoDataFrame
        Merged GeoDataFrame containing county boundaries and VALUE colum
    map_object : folium.Map
        Existing folium map object to add the layer to
    caption : str
        Legend caption describing the layer e.g. 
        'Total Afforestation 2023 (Hectares)'
    name : str
        Layer name shown in the layer toggle control
    cmap : str
        Matplotlib colormap to use. Default is 'YlGn'
    
    Returns:
    --------
    folium.Map
        Map object with the new choropleth layer added
    """
    # Add folium layer to existing map
    merged_gdf.explore('VALUE',
                       m=map_object,
                       cmap=cmap,
                       style_kwds={'fillOpacity': 0.7},
                       legend=True,
                       legend_kwds={'caption': caption},
                       name=name,
                       )
    return map_object

In [ ]:
# Create base map
m = folium.Map(location=[53.5, -8], 
               zoom_start=7, 
               tiles='CartoDB positron')

In [ ]:
# Add all three afforestation layers to map
add_folium_layer(merged_total, m, 
                 'Total Afforestation 2023 (Hectares)', 
                 'Total Afforestation')
add_folium_layer(merged_broadleaf, m, 
                 'Broadleaf Afforestation 2023 (Hectares)', 
                 'Broadleaf')
add_folium_layer(merged_conifer, m, 
                 'Conifer Afforestation 2023 (Hectares)', 
                 'Conifer')

In [ ]:
# Add layer toggle control and display map
folium.LayerControl().add_to(m)
m

In [ ]:
# Save map to html
m.save('ireland_woodlandmap.html')